## Transformez les variables pour faciliter l’apprentissage du modèle

Le traitement apporté aux données catégoriques textuelles va dépendre du nombre de catégories prises par la variable. On distinguera :

- le cas binaire où l'on a 2 catégories ;

- et pour être très précis, les cas "un peu plus que 2" ou "carrément beaucoup".

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
dataset_url = "https://raw.githubusercontent.com/OpenClassrooms-Student-Center/8063076-Initiez-vous-au-Machine-Learning/master/data/paris-arbres-clean-2023-09-10.csv"
data = pd.read_csv(dataset_url)
df = data[data.libelle_francais == "Platane"].copy()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_6868\2511565868.py:2: DtypeWarning: Columns (0: idbase, 1: numero) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(dataset_url)


In [4]:
df.head()

,idbase,domanialite,arrondissement,complement_adresse,numero,lieu_adresse,idemplacement,libelle_francais,genre,espece,variete_oucultivar,circonference_cm,hauteur_m,stade_de_developpement,remarquable,geo_point_2d
3,238226,Alignement,PARIS 12E ARRDT,37,NaN,AVENUE DE SAINT MANDE,000501020,Platane,Platanus,x hispanica,NaN,30.0,5.0,Jeune (arbre),NON,"48.84515461889761, 2.4008303350818525"
9,289464,Alignement,PARIS 12E ARRDT,NaN,NaN,AVENUE LEDRU ROLLIN,000402007,Platane,Platanus,x hispanica,NaN,60.0,10.0,Adulte,NON,"48.84717970255216, 2.37017013054543"
10,221701,Alignement,PARIS 16E ARRDT,NaN,NaN,BOULEVARD DE L AMIRAL BRUIX,000705007,Platane,Platanus,x hispanica,NaN,68.0,10.0,Jeune (arbre)Adulte,NON,"48.87428152523608, 2.2783797113253454"
11,227024,Alignement,PARIS 19E ARRDT,59,NaN,RUE DAVID D ANGERS,000901019,Platane,Platanus,x hispanica,''Pyramidalis'',30.0,5.0,Jeune (arbre),NON,"48.881527844507275, 2.395607068367473"
17,285387,Alignement,PARIS 10E ARRDT,face à peugeot à gauche des wc,NaN,RUE DU FAUBOURG SAINT MARTIN,001301027,Platane,Platanus,x hispanica,NaN,53.0,8.0,Adulte,NON,"48.878100132217135, 2.3616292876411995"


## Le cas binaire
La variable prend 2 valeurs distinctes et exclusives.

Sur le dataset des arbres, la variable  remarquable  est binaire.

In [5]:
df.remarquable.value_counts()

remarquable
NON    41316
OUI       36
Name: count, dtype: int64

On va maintenant transformer ces valeurs en chiffre

In [11]:
df.info()

<class 'pandas.DataFrame'>
Index: 42588 entries, 3 to 221193
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   idbase                  42588 non-null  object 
 1   domanialite             42588 non-null  str    
 2   arrondissement          42588 non-null  str    
 3   complement_adresse      15309 non-null  str    
 4   numero                  0 non-null      str    
 5   lieu_adresse            42588 non-null  str    
 6   idemplacement           42588 non-null  str    
 7   libelle_francais        42588 non-null  str    
 8   genre                   42588 non-null  str    
 9   espece                  42584 non-null  str    
 10  variete_oucultivar      260 non-null    str    
 11  circonference_cm        42588 non-null  float64
 12  hauteur_m               42588 non-null  float64
 13  stade_de_developpement  39238 non-null  str    
 14  remarquable             41352 non-null  str    
 15  

In [ ]:
df.remarquable = df.remarquable.replace({"NON":0,"OUI":1})
df.remarquable.value_counts()

remarquable
0    41316
1       36
Name: count, dtype: int64

## Cas non binaire mais peu de catégories

On distingue :

- les catégories ordonnées (ordinales) ;

- des catégories non ordonnées.

## L'encodage ordonné
On peut remplacer les catégories ordinales par des nombres croissants sans modifier la quantité d'informations contenues dans le jeu de données.

In [18]:
from category_encoders.ordinal import OrdinalEncoder

mapping = [
    {
        "col": "stade_de_developpement",
        "mapping": {
            np.nan: 0,
            "Jeune (arbre)": 1,
            "Jeune (arbre)Adulte": 2,
            "Adulte": 3,
            "Mature": 4,
        },
    }
]

encoder = OrdinalEncoder(mapping=mapping)
stade = encoder.fit_transform(df.stade_de_developpement)
stade.value_counts()

stade_de_developpement
3                         21620
2                          8356
1                          5916
0                          3350
4                          3346
Name: count, dtype: int64

In [19]:
df['stade_de_developpement']=stade.copy()

In [20]:
df["stade_de_developpement"].value_counts(dropna=False)

stade_de_developpement
3    21620
2     8356
1     5916
0     3350
4     3346
Name: count, dtype: int64

## Cas non ordonné 

### L'encodage one-hot

Le  principe est d'associer pour chaque valeur de la catégorie une nouvelle variable binaire qui indique si la variable originale avait la valeur en question. Si la variable prend N valeurs distinctes, on introduira donc N-1 nouvelles variables, la Nième étant redondante.

 le one hot encoding n'est réellement utilisable que lorsque le nombre de valeurs de la variable à numériser est faible par rapport aux nombre de variables utiles du dataset.

In [22]:
# Limitons nous à 3 valeurs
df = df[df.domanialite.isin(["Alignement", "Jardin", "CIMETIERE"])].copy()
df.reset_index(drop=True, inplace=True)

In [24]:
from sklearn.preprocessing import OneHotEncoder

enc = OneHotEncoder()
labels = enc.fit_transform(df.domanialite.values.reshape(-1, 1)).toarray()

In [25]:
df = pd.concat([df, pd.DataFrame(columns=["Alignement", "Jardin"],data=labels[:,:2])],axis=1)

## L'encodage Binaire

L'énorme avantage de l’encodage binaire sur le one hot encoding est la réduction drastique du nombre de variables nécessaires pour encoder la variable catégorique originale.

On perd par contre la correspondance directe entre la valeur de la variable booléenne qui ne représente plus qu'un des digits de l'encodage, et la catégorie originale.



In [31]:
from category_encoders.binary import BinaryEncoder

enc = BinaryEncoder()
espece_bin = enc.fit_transform(df.espece)
espece_bin.shape

(40906, 3)

In [32]:
espece_bin.head()

,espece_0,espece_1,espece_2
0,0,0,1
1,0,0,1
2,0,0,1
3,0,0,1
4,0,0,1


## Le Feature Engineering (création de nouvelles variables)

l'amélioration du dataset par le feature engineering revient à faire à un véritable travail de détective à partir :

- de l'analyse statistique des variables ;

- des connaissances du domaine ;

- de l'intégration de datasets externes ;

- de l'étude précise des erreurs de prédiction du modèle.

Techniques répandues :

- la mise en intervalle (bucket) des données continues ;

- la transformation de variables existantes : puissance, inverse, racine carrée ou log, etc. ;

- la création d’une variable  flag  qui indique si une autre variable est renseignée ou manquante, ou tout autre caractéristique notable ;

- et la  compilation de plusieurs variables ensemble : par opérations sur les variables numériques, agrégation de textes, filtres sur les images, etc.